# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [15]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [16]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [43]:
spaceship["HomePlanet"].value_counts()

HomePlanet
Earth     3566
Europa    1673
Mars      1367
Name: count, dtype: int64

In [17]:
spaceship.shape

(8693, 14)

**Check for data types**

In [ ]:
spaceship.dtypes

**Check for missing values**

In [ ]:
spaceship.isnull().sum()

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [20]:
spaceship = spaceship.dropna()

In [ ]:
spaceship.isnull().sum()

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [22]:
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]

In [ ]:
spaceship["Cabin"].value_counts()

- Drop PassengerId and Name

In [24]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

In [ ]:
spaceship

- For non-numerical columns, do dummies.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
categorical_columns = ["HomePlanet","CryoSleep","Destination","Cabin","VIP","Transported"]

X_cat = spaceship[categorical_columns]



encoder = OneHotEncoder(drop='first').fit(X_cat)
cols = encoder.get_feature_names_out(input_features=X_cat.columns)
X_cat_encode = pd.DataFrame(encoder.transform(X_cat).toarray(),columns=cols)
X_cat_encode.head()

In [ ]:
spaceship = spaceship.reset_index(drop=True)
X_cat_encode = X_cat_encode.reset_index(drop=True)

spaceship_encode = pd.concat([spaceship, X_cat_encode], axis=1)
spaceship_encode=spaceship_encode.drop(columns=["HomePlanet","CryoSleep","Destination","Cabin","VIP","Transported"])
spaceship_encode.head()

**Perform Train Test Split**

In [51]:
features = spaceship_encode.drop(columns = ["Age"])
target = spaceship_encode["Age"]

In [52]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [53]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
normalizer = MinMaxScaler()
normalizer.fit(X_train)

X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)

X_train_norm = pd.DataFrame(X_train_norm, columns = X_train.columns)
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test.columns)

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

- Evaluate your model's performance. Comment it

In [96]:
knn = KNeighborsRegressor(n_neighbors=10)
knn.fit(X_train_norm, y_train)

knn.score(X_test_norm, y_test)

0.09239265945583708

In [97]:
y_pred = knn.predict(X_test_norm)

In [98]:
from sklearn.metrics import (r2_score,mean_absolute_error, mean_squared_error, root_mean_squared_error)

print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

R²: 0.09239265945583708
MAE: 10.686157337367623
MSE: 190.13713313161875
RMSE: 13.789022196356736
